# Baseline pipeline (v2 — label-mapping fix)

Same TF-IDF + classical-model baseline as `baseline_pipeline_notebook.ipynb`, with the
label-mapping bug fixed (see `claude-workspace/ISSUE_PLAN.md`, issue I-1) and a corrected
evaluation: macro-F1 as the primary metric, per-class precision/recall/F1, confusion
matrices, and Dummy baselines (I-2, I-5). Labels and evaluation come from `liar_utils.py`
so the baseline and proposed pipelines can never diverge (I-4).

**2026-08-24 update (train+valid merge):** `valid.csv` was previously loaded and scored
alongside test but never used for any model-selection decision -- a wasted split. It is
now merged into the training pool (`train_full = train + valid`) before fitting, so every
classical model gets ~1,284 more labeled rows. Test stays untouched and is the only
held-out split reported. The DistilBERT reference notebook is deliberately *not* changed
to match -- it still follows the official train/valid/test split, so its training-data
budget differs from the classical models' here; this is disclosed in the paper rather than
silently left implicit.

In [1]:
import re

import pandas as pd
from nltk.corpus import stopwords
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

from liar_utils import RANDOM_STATE, evaluate_full, load_and_label, print_report

Load data with the corrected label mapping

In [2]:
train_df = load_and_label("train.csv")
valid_df = load_and_label("valid.csv")
test_df = load_and_label("test.csv")

# Merge train+valid into one fitting pool (see the 2026-08-24 note above); test
# stays untouched and is the only held-out split reported below.
train_full = pd.concat([train_df, valid_df], ignore_index=True)

balance = train_full["Label"].value_counts(normalize=True).rename({0: "fake", 1: "real"})
print("Train(+valid) class balance:\n", balance)
fake_share = balance["fake"]
assert 0.35 < fake_share < 0.5, "Fake-class share outside the expected ~44% range -- check label mapping"

Train(+valid) class balance:
 Label
real    0.557098
fake    0.442902
Name: proportion, dtype: float64


Preprocess text

In [3]:
stop_words = set(stopwords.words("english"))


def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    tokens = text.split()
    tokens = [w for w in tokens if w not in stop_words]
    return " ".join(tokens)


train_full["clean_text"] = train_full["Statement"].apply(preprocess_text)
test_df["clean_text"] = test_df["Statement"].apply(preprocess_text)

TF-IDF feature extraction

In [4]:
X_train = train_full["clean_text"]
X_test = test_df["clean_text"]

y_train = train_full["Label"]
y_test = test_df["Label"]

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

Dummy baselines (I-5) -- the bar every real model must clear

In [5]:
dummy_most_frequent = DummyClassifier(strategy="most_frequent")
dummy_stratified = DummyClassifier(strategy="stratified", random_state=RANDOM_STATE)

dummy_most_frequent.fit(X_train_tfidf, y_train)
dummy_stratified.fit(X_train_tfidf, y_train)

DummyClassifier(random_state=42, strategy='stratified')

Models -- including class_weight='balanced' variants of LR/SVM (I-5)

In [6]:
models = {
    "Dummy (most frequent)": dummy_most_frequent,
    "Dummy (stratified)": dummy_stratified,
    "Naive Bayes": MultinomialNB().fit(X_train_tfidf, y_train),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE).fit(
        X_train_tfidf, y_train
    ),
    "Logistic Regression (balanced)": LogisticRegression(
        max_iter=1000, random_state=RANDOM_STATE, class_weight="balanced"
    ).fit(X_train_tfidf, y_train),
    "SVM": LinearSVC(random_state=RANDOM_STATE).fit(X_train_tfidf, y_train),
    "SVM (balanced)": LinearSVC(random_state=RANDOM_STATE, class_weight="balanced").fit(
        X_train_tfidf, y_train
    ),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE).fit(
        X_train_tfidf, y_train
    ),
}

Evaluate -- macro-F1 primary, per-class P/R/F1, confusion matrix (I-2)

In [7]:
rows = []
for name, model in models.items():
    y_test_pred = model.predict(X_test_tfidf)
    print_report(name, y_test, y_test_pred)

    test_metrics = evaluate_full(y_test, y_test_pred)
    rows.append(
        {
            "Pipeline": "Baseline",
            "Method": "TF-IDF only",
            "Model": name,
            "Test Accuracy": test_metrics["accuracy"],
            "Test Macro-F1": test_metrics["macro_f1"],
            "Test Fake Precision": test_metrics["fake_precision"],
            "Test Fake Recall": test_metrics["fake_recall"],
            "Test Fake F1": test_metrics["fake_f1"],
            "Test Real F1": test_metrics["real_f1"],
            "Test Confusion Matrix": test_metrics["confusion_matrix"],
        }
    )

baseline_results = pd.DataFrame(rows)
baseline_results


Dummy (most frequent)
[[  0 553]
 [  0 714]]
              precision    recall  f1-score   support

        fake      0.000     0.000     0.000       553
        real      0.564     1.000     0.721       714

    accuracy                          0.564      1267
   macro avg      0.282     0.500     0.360      1267
weighted avg      0.318     0.564     0.406      1267


Dummy (stratified)
[[239 314]
 [327 387]]
              precision    recall  f1-score   support

        fake      0.422     0.432     0.427       553
        real      0.552     0.542     0.547       714

    accuracy                          0.494      1267
   macro avg      0.487     0.487     0.487      1267
weighted avg      0.495     0.494     0.495      1267


Naive Bayes
[[234 319]
 [171 543]]
              precision    recall  f1-score   support

        fake      0.578     0.423     0.489       553
        real      0.630     0.761     0.689       714

    accuracy                          0.613      1267
   

/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{m

,Pipeline,Method,Model,Test Accuracy,Test Macro-F1,Test Fake Precision,Test Fake Recall,Test Fake F1,Test Real F1,Test Confusion Matrix
0,Baseline,TF-IDF only,Dummy (most frequent),0.563536,0.360424,0.000000,0.000000,0.000000,0.720848,"[[0, 553], [0, 714]]"
1,Baseline,TF-IDF only,Dummy (stratified),0.494081,0.487082,0.422261,0.432188,0.427167,0.546996,"[[239, 314], [327, 387]]"
2,Baseline,TF-IDF only,Naive Bayes,0.613260,0.588802,0.577778,0.423146,0.488518,0.689086,"[[234, 319], [171, 543]]"
3,Baseline,TF-IDF only,Logistic Regression,0.624309,0.604582,0.589327,0.459313,0.516260,0.692903,"[[254, 299], [177, 537]]"
4,Baseline,TF-IDF only,Logistic Regression (balanced),0.605367,0.601488,0.544992,0.580470,0.562172,0.640805,"[[321, 232], [268, 446]]"
5,Baseline,TF-IDF only,SVM,0.599842,0.590235,0.544231,0.511754,0.527493,0.652977,"[[283, 270], [237, 477]]"
6,Baseline,TF-IDF only,SVM (balanced),0.583268,0.578902,0.521368,0.551537,0.536028,0.621777,"[[305, 248], [280, 434]]"
7,Baseline,TF-IDF only,Random Forest,0.612470,0.589127,0.575243,0.428571,0.491192,0.687062,"[[237, 316], [175, 539]]"


In [8]:
baseline_results.to_csv("baseline_results_v2.csv", index=False)